In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from dscript import FullyConnectedEmbed, ContactCNN, ModelInteraction
from dscript import PairedDataset
from dscript import collate_paired_sequences
from dscript import interaction_grad, interaction_eval


In [ ]:
spe = "yeast"

data_dir = "ppi-data"
train_file = os.path.join(data_dir, spe, "action/train_action_20.tsv")
val_file = os.path.join(data_dir, spe, "action/test_action_20.tsv")
epochs = 10

# from google.colab import drive
#
# drive.mount('/content/drive')
# data_dir = "drive/MyDrive/ppi-data"
# train_file = os.path.join(data_dir, spe, "action/train_action.tsv")
# val_file = os.path.join(data_dir, spe, "action/test_action.tsv")
# epochs = 100

embedding_h5 = os.path.join(data_dir, spe, "seq/pipr.embedding.h5")

input_dim = 13
embed_dim = 26

num_epochs = 50
batch_size = 32
lr = 0.001,

out_dir = "result/dscript"
save_model = False
seed = 1234

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

use_cuda = torch.cuda.is_available()

train_df = pd.read_csv(train_file, sep="\t", header=None)
train_dataset = PairedDataset(train_df[0].to_list(), train_df[1].to_list(),
                              train_df[2].to_list(), embedding_h5)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_paired_sequences,
    shuffle=True,
)

val_df = pd.read_csv(val_file, sep="\t", header=None)
val_dataset = PairedDataset(val_df[0].to_list(), val_df[1].to_list(),
                            val_df[2].to_list(), embedding_h5)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=collate_paired_sequences
)

In [ ]:
embedding_model = FullyConnectedEmbed(input_dim, embed_dim)
contact_model = ContactCNN(embed_dim)
model = ModelInteraction(embedding_model, contact_model, use_cuda)

if use_cuda:
    model.cuda()

# digits = int(np.floor(np.log10(num_epochs))) + 1

params = [p for p in model.parameters() if p.requires_grad]
optim = torch.optim.Adam(params, lr=lr)

In [ ]:

epoch_report_fmt = "Finished Epoch {}/{}: Loss={:.6}, Accuracy={:.3%}, MSE={:.6}, Precision={:.6}, Recall={:.6}, F1={:.6}, AUPR={:.6}"

N = len(train_loader) * batch_size

for epoch in range(num_epochs):
    model.train()

    n = 0
    loss_accum = 0
    acc_accum = 0
    mse_accum = 0

    for x1, x2, y in train_loader:
        loss, correct, mse, b = interaction_grad(model, x1, x2, y, use_cuda=use_cuda)

        n += b
        delta = b * (loss - loss_accum)
        loss_accum += delta / n

        delta = correct - b * acc_accum
        acc_accum += delta / n

        delta = b * (mse - mse_accum)
        mse_accum += delta / n

        optim.step()
        optim.zero_grad()
        model.clip()

    model.eval()

    with torch.no_grad():
        (val_loss,
         val_correct,
         val_mse,
         val_pr,
         val_re,
         val_f1,
         val_aupr) = interaction_eval(model, val_loader, use_cuda)

        tokens = [
            epoch + 1,
            num_epochs,
            val_loss,
            val_correct / (len(val_loader) * batch_size),
            val_mse,
            val_pr,
            val_re,
            val_f1,
            val_aupr,
        ]
        print(epoch_report_fmt.format(*tokens))

if save_model is not None:
    save_path = os.path.join(out_dir, "final.sav")
    print(f"Saving final model to {save_path}")
    model.cpu()
    torch.save(model, save_path)
    if use_cuda:
        model.cuda()